In [1]:
import numpy as np

#### Задача 1 
#### Исследовать ограничения чисел с плавающей точкой на примере float32 и float64

In [5]:
# 1.1 Сравнил массив float с точным значением
arr = np.array([0.1, 0.2, 0.3], dtype=np.float32)
s = np.sum(arr)
diff = s - 0.6
print("Сумма:", s, "Разница:", diff)

# 1.2 Машинный эпсилон для float32 и float64
eps32 = np.finfo(np.float32).eps
eps64 = np.finfo(np.float64).eps
print("Эпсилон для float32:", eps32)
print("Эпсилон для float64:", eps64)

# 1.3 Эффект потери точности
a = np.float32(1.0000001)
b = np.float32(1.0000000)
res = a - b
print("Разность:", res)

Сумма: 0.6 Разница: 0.0
Эпсилон для float32: 1.1920929e-07
Эпсилон для float64: 2.220446049250313e-16
Разность: 1.1920929e-07


#### Задача 2
#### Освоить различные способы индексации и правила broadcasting

In [11]:
# 2.1 Создал массив A
A = np.random.randint(-5, 10, size=(4, 5))
print("Исходный массив A:\n", A)

sec_row = A[1, :]
col_3 = A[:, 3]
sub_arr = A[1:4, 2:5]

print("Вторая строка:", sec_row)
print("Столбец с индексом 3:", col_3)
print("Подмассив:\n", sub_arr)
print('\n')

# 2.2 Взял срез с помощью fancy-индексации
rows = [0, 2, 3]
cols = [1, 4]
fancy_res = A[np.ix_(rows, cols)]
print("Срез по индексам и строкам:\n", fancy_res)
print('\n')

# 2.3 Заменил отрицательные на 0
A[A < 0] = 0
print("После замены отрицательных на 0:\n", A)
print('\n')

# 2.4 broadcasting
B = np.array([[1], [2], [3], [4]])
C = np.array([[10, 20, 30, 40, 50]])
sum_bc = B + C
print("Результат сложения B и C:\n", sum_bc)

Исходный массив A:
 [[ 2 -1  1  3 -2]
 [-4 -3  9 -1 -2]
 [ 3 -4  1  6  8]
 [ 9  1  9  5  0]]
Вторая строка: [-4 -3  9 -1 -2]
Столбец с индексом 3: [ 3 -1  6  5]
Подмассив:
 [[ 9 -1 -2]
 [ 1  6  8]
 [ 9  5  0]]


Срез по индексам и строкам:
 [[-1 -2]
 [-4  8]
 [ 1  0]]


После замены отрицательных на 0:
 [[2 0 1 3 0]
 [0 0 9 0 0]
 [3 0 1 6 8]
 [9 1 9 5 0]]


Результат сложения B и C:
 [[11 21 31 41 51]
 [12 22 32 42 52]
 [13 23 33 43 53]
 [14 24 34 44 54]]


#### Задача 3
#### Научиться работать с массивами, не помещающимися в память, и понять влияние порядка доступа на производительность

In [21]:
# 3.1 Создал memory-mapped файл
filename = "data.dat"
arr = np.memmap(filename, dtype=np.float64, mode="w+", shape=(1000, 1000))
arr[:] = np.random.rand(1000, 1000)
arr.flush()

# 3.2 Прочитал отдельный столбец и строку
arr_read = np.memmap(filename, dtype=np.float64, mode="r", shape=(1000, 1000))
row_5 = arr_read[5, :]
col_10 = arr_read[:, 10]
print("Первые 5 элементов 5 строки:", row_5[:5])
print("Первые 5 элементов 10 столбца:", col_10[:5])

# 3.3 Сравнил время обхода по строкам и столбцам
def sum_by_rows(a):
    total = 0.0
    for i in range(a.shape[0]):
        total += np.sum(a[i, :])
    return total

def sum_by_cols(a):
    total = 0.0
    for j in range(a.shape[1]):
        total += np.sum(a[:, j])
    return total

print("Обход по строкам:")
%timeit sum_by_rows(arr_read)

print("Обход по столбцам:")
%timeit sum_by_cols(arr_read)

print('\n')
print(f"""Обход по строкам быстрее (3.01 мс против 3.33 мс), потому что в NumPy числа одной строки лежат в памяти подряд — 
процессор считывает их за один заход. А чтобы собрать столбец, ему приходится  каждый раз перепрыгивать 
через всю строку, поэтому получается медленнее.""")

Первые 5 элементов 5 строки: [0.50635199 0.57852462 0.94916254 0.07072618 0.52914808]
Первые 5 элементов 10 столбца: [0.76474671 0.62131436 0.53194377 0.71673537 0.89157989]
Обход по строкам:
3.01 ms ± 17.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Обход по столбцам:
3.33 ms ± 5.44 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Обход по строкам быстрее (3 мс против 3,34 мс), потому что в NumPy числа одной строки лежат в памяти подряд — 
процессор считывает их за один заход. А чтобы собрать столбец, ему приходится  каждый раз перепрыгивать 
через всю строку, поэтому получается медленнее.


#### Задача 4
#### Изучить форматы хранения разреженных матриц и операции с ними

In [24]:
from scipy import sparse

# 4.1 Создал разреженную матрицу
mat_csr = sparse.random(5, 5, density=0.2, format="csr", random_state=42)
print("Разреженная матрица:\n", mat_csr.toarray())

# 4.2 Преобразовал в COO и CSC
mat_coo = mat_csr.tocoo()
mat_csc = mat_csr.tocsc()
print("Формат COO:\n", mat_coo)
print("Формат CSC:\n", mat_csc)

# 4.3 Умножил матрицу на вектор
vec = np.array([1, 2, 3, 4, 5])
res_vec = mat_csr.dot(vec)
print("Результат умножения на вектор:", res_vec)

# 4.4 Замерил скорость на большой таблице
big_sparse = sparse.random(1000, 1000, density=0.01, format="csr", random_state=42)
big_dense = big_sparse.toarray()

print("Умножение разреженных матриц:")
%timeit big_sparse.dot(big_sparse)

print("Умножение плотных матриц:")
%timeit big_dense.dot(big_dense)

Разреженная матрица:
 [[0.43194502 0.         0.         0.         0.        ]
 [0.         0.         0.61185289 0.52475643 0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.30424224 0.         0.         0.29122914]
 [0.         0.         0.         0.         0.        ]]
Формат COO:
 <COOrdinate sparse matrix of dtype 'float64'
	with 5 stored elements and shape (5, 5)>
  Coords	Values
  (0, 0)	0.43194501864211576
  (1, 2)	0.6118528947223795
  (1, 3)	0.5247564316322378
  (3, 1)	0.3042422429595377
  (3, 4)	0.2912291401980419
Формат CSC:
 <Compressed Sparse Column sparse matrix of dtype 'float64'
	with 5 stored elements and shape (5, 5)>
  Coords	Values
  (0, 0)	0.43194501864211576
  (3, 1)	0.3042422429595377
  (1, 2)	0.6118528947223795
  (1, 3)	0.5247564316322378
  (3, 4)	0.2912291401980419
Результат умножения на вектор: [0.43194502 3.93458441 0.         2.06463019 0.        ]
Умножение разреженных матриц:
573 μs ± 8.87 μs per loop (mean ± std. de

#### Задача 5
#### Познакомиться с ускорением циклов с помощью Cython

In [25]:
# Сумма квадратов элементов списка
def sum_squares_py(lst):
    total = 0.0
    for x in lst:
        total += x * x
    return total

In [ ]:
# Компилирую файл .pyx

In [29]:
!python3 setup.py build_ext --inplace

Compiling fast_sum.pyx because it changed.
[1/1] Cythonizing fast_sum.pyx
running build_ext
building 'fast_sum' extension
creating build/temp.macosx-12.1-arm64-cpython-313
clang -fno-strict-overflow -Wsign-compare -Wunreachable-code -DNDEBUG -O2 -Wall -fPIC -O2 -isystem /Users/aleksandr2006/miniconda3/include -arch arm64 -fPIC -O2 -isystem /Users/aleksandr2006/miniconda3/include -arch arm64 -I/Users/aleksandr2006/miniconda3/include/python3.13 -c fast_sum.c -o build/temp.macosx-12.1-arm64-cpython-313/fast_sum.o
fast_sum.c:22542:50: warning: code will never be executed [-Wunreachable-code]
 22542 |     if (unlikely(__Pyx_PyUnicode_READY(s1) < 0)) goto bad;
       |                                                  ^~~~~~~~
1 warning generated.
creating build/lib.macosx-12.1-arm64-cpython-313
clang -bundle -undefined dynamic_lookup -Wl,-rpath,/Users/aleksandr2006/miniconda3/lib -L/Users/aleksandr2006/miniconda3/lib -Wl,-rpath,/Users/aleksandr2006/miniconda3/lib -L/Users/aleksandr2006/minic

In [ ]:
# Сравнил время выполнения обеих функций

In [30]:
from fast_sum import sum_squares_cy

arr = np.random.rand(10_000_000)

print("Обычный Python:")
%timeit sum_squares_py(arr)

print("Cython:")
%timeit sum_squares_cy(arr)

Обычный Python:
700 ms ± 6.98 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
Cython:
12.6 ms ± 78.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
